# IndicOOC — Encoder Transfer Experiment (Objective O2-iii)

Notebooks 01 and 04 both assume one encoder: the English-trained `google/siglip-base-patch16-224`. Section 5.2 of the synopsis names that as an **untested assumption** — SigLIP was trained overwhelmingly on English image–text pairs, so its scores on Devanagari/Bengali/Marathi claims may be systematically worse, not just noisier.

This notebook tests the assumption instead of carrying it. The same images and the same claims are scored by three encoders:

| key | checkpoint | trained on |
|---|---|---|
| `siglip_en` | `google/siglip-base-patch16-224` | predominantly English |
| `siglip_multi` | `google/siglip-base-patch16-256-multilingual` | multilingual WebLI |
| `siglip2` | `google/siglip2-base-patch16-224` | multilingual, newer recipe |

Nothing but the encoder changes, so any difference is a transfer effect.

**Headline metric is matched-pair ranking accuracy, not macro-F1.** Your pilot has minimal pairs — the same image with a true and a false caption. Asking "does the encoder rank the true caption above the false one for this image" is threshold-free and paired, which makes it far more stable at pilot scale than thresholding at an arbitrary 0.5. macro-F1 is reported alongside it, with its sample size attached.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers accelerate pillow pandas scikit-learn sentencepiece matplotlib

## Step 1 — Load the pilot set with the same corrections as notebook 01

Identical override and exclusion logic, so this experiment sits on the same data as every other baseline.

In [ ]:
import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/IndicOOC'
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'app'))

import pandas as pd
from scoring import ENCODERS, ENCODER_LABELS, load_image_from_url, score_pair, classify

df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/annotations/pilot_v0.csv'))

LANGUAGE_OVERRIDES = {'hi_ooc_002': 'mr'}
EXCLUDE_IDS = {'mr_ooc_002'}

df['claim_language'] = df.apply(lambda r: LANGUAGE_OVERRIDES.get(r['id'], r['language']), axis=1)
eval_df = df[~df['id'].isin(EXCLUDE_IDS)].copy()

print(f'{len(eval_df)} rows in scope')
print(eval_df['claim_language'].value_counts())

## Step 2 — Download every image once

Images are fetched once and reused across all three encoders. Scoring the same bytes everywhere is what makes the comparison fair — a download that succeeds for one encoder and fails for another would silently change the sample.

In [ ]:
images = {}
for _, row in eval_df.iterrows():
    img = load_image_from_url(row['image_url'])
    if img is None:
        print(f'  [skip] {row["id"]}')
    images[row['id']] = img

usable = eval_df[eval_df['id'].map(lambda i: images[i] is not None)].copy()
print(f'{len(usable)}/{len(eval_df)} rows usable')

## Step 3 — Score every row with every encoder

First run downloads each checkpoint (~1.5 GB each). Expect a few minutes and roughly 5 GB of disk; all three fit comfortably in a T4's memory because they are loaded and used sequentially.

In [ ]:
records = []
for enc in ENCODERS:
    print(f'--- {ENCODER_LABELS[enc]} ---')
    for _, row in usable.iterrows():
        score = score_pair(images[row['id']], row['claim_text'], encoder=enc)
        records.append({
            'encoder': enc,
            'id': row['id'],
            'image_url': row['image_url'],
            'claim_language': row['claim_language'],
            'label': row['label'],
            'score': score,
            'pred_label': classify(score),
        })
    print('done')

scores_df = pd.DataFrame(records)
scores_df.head()

## Step 4 — Matched-pair ranking accuracy per encoder and language

For every image that has both a `supported` and an `out_of_context` claim, the encoder is correct if it scores the supported one higher. No threshold involved.

In [ ]:
def pair_results(group):
    out = []
    for url, g in group.groupby('image_url'):
        if set(g['label']) >= {'supported', 'out_of_context'}:
            sup = g[g['label'] == 'supported']['score'].iloc[0]
            ooc = g[g['label'] == 'out_of_context']['score'].iloc[0]
            out.append({
                'image_url': url,
                'claim_language': g['claim_language'].iloc[0],
                'correct': bool(sup > ooc),
                'margin': sup - ooc,
            })
    return pd.DataFrame(out)

pair_rows = []
for enc, g in scores_df.groupby('encoder'):
    pr = pair_results(g)
    if len(pr) == 0:
        continue
    pr['encoder'] = enc
    pair_rows.append(pr)

pairs_df = pd.concat(pair_rows) if pair_rows else pd.DataFrame()
if len(pairs_df):
    summary = (pairs_df.groupby('encoder')
               .agg(pairs=('correct', 'size'),
                    ranked_correct=('correct', 'sum'),
                    mean_margin=('margin', 'mean'))
               .reset_index())
    summary['pair_accuracy'] = (summary['ranked_correct'] / summary['pairs']).round(3)
    summary['mean_margin'] = summary['mean_margin'].round(4)
    display(summary)
else:
    print('No complete matched pairs in the usable set.')

## Step 5 — Macro-F1 per encoder and language, with n attached

Reported for completeness and because Review 1 asks for the metric by name. Read it with the sample size next to it — at this scale one row flipping moves it a long way.

In [ ]:
from sklearn.metrics import f1_score

rows = []
for enc, g in scores_df.groupby('encoder'):
    for lang, gl in g.groupby('claim_language'):
        single_class = gl['label'].nunique() < 2
        rows.append({
            'encoder': ENCODER_LABELS[enc],
            'claim_language': lang,
            'n': len(gl),
            'macro_f1': None if single_class else round(
                f1_score(gl['label'], gl['pred_label'], average='macro', zero_division=0), 3),
            'accuracy': round((gl['label'] == gl['pred_label']).mean(), 3),
            'mean_score_supported': round(gl[gl['label'] == 'supported']['score'].mean(), 4),
            'mean_score_ooc': round(gl[gl['label'] == 'out_of_context']['score'].mean(), 4),
        })

encoder_comparison = pd.DataFrame(rows).sort_values(['claim_language', 'encoder'])
os.makedirs(os.path.join(PROJECT_ROOT, 'outputs'), exist_ok=True)
encoder_comparison.to_csv(os.path.join(PROJECT_ROOT, 'outputs/encoder_comparison.csv'), index=False)
encoder_comparison

## Step 6 — Separation plot

The distance between the mean supported score and the mean out-of-context score is the quantity a threshold has to sit inside. A separation near zero means no threshold can work, whatever value you pick — which is a more useful diagnosis than a bad F1.

In [ ]:
import matplotlib.pyplot as plt

sep = encoder_comparison.dropna(subset=['mean_score_supported', 'mean_score_ooc']).copy()
sep['separation'] = sep['mean_score_supported'] - sep['mean_score_ooc']

fig, ax = plt.subplots(figsize=(8, 4))
for enc, g in sep.groupby('encoder'):
    ax.bar([f'{l}\n{enc.split()[-1]}' for l in g['claim_language']], g['separation'], label=enc)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('mean(supported) - mean(out_of_context)')
ax.set_title('Score separation by encoder and claim language')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs/encoder_separation.png'), dpi=150)
plt.show()

## How to present this

Say what it is: **a transfer experiment whose machinery is complete and whose sample size is not**. At pilot scale the ranking of the three encoders is not yet stable, and claiming one is better would not survive a follow-up question.

What it does establish for Review 1:

- The Section 5.2 transfer assumption is now an experiment with code behind it, not an assumption carried forward
- Three encoders are evaluated on identical images, identical claims and a fixed split
- The metric chosen (matched-pair ranking) is the one that stays valid at small n, and macro-F1 is reported with its sample size rather than alone
- Scaling the dataset changes the input to this notebook and nothing else

The honest sentence for the slide: *"the comparison framework is built and runs; the numbers become trustworthy at a few hundred test rows, which is the Review 2 milestone."*